# Demonstration of unconditional generation
Unconditional generation is only used during validation and approximates the ability for the model to learn the distribution.

In [ ]:
from rdkit import Chem
import torch

from shepherd import load_model
from shepherd.extract import mol_charges_from_samples

from shepherd_score.visualize import draw_sample, draw_mol

# faster inference
torch.set_float32_matmul_precision("high")

### Load MOSES model
Downloads from Huggingface automatically.

In [ ]:
model = load_model(
    model_type='mosesaq',
    inference_only=True,
    device='cuda'
)

### Generate
Generate samples unconditionally. Adjust `batch_size`, `N_x1`, and `N_x4` as you'd like. 

In [ ]:
unconditional_samples = model.generate(
    batch_size = 48, # tune based on GPU
    N_x1 = 40, # number of atoms in the sample
    N_x4 = 6, # number of pharmacophores in the sample
)

### Visualize results

`unconditional_samples` is a `GeneratedSample` class which allows users to index into the generated samples

In [ ]:
# index into the list to get a sample
# x1 (atoms)
print(f'atom types: {unconditional_samples[0].atoms}')
print(f'atom pos (shape): {unconditional_samples[0].positions.shape}')
# x2 (surface)
print(f'surface (shape): {unconditional_samples[0].surface.shape}')
# x3 (electrostatics)
print(f'electrostatics (shape): {unconditional_samples[0].electrostatics.shape}')
# x4 (pharmacophores)
print(f'pharm types: {unconditional_samples[0].pharm_types}')
print(f'pharm pos (shape): {unconditional_samples[0].pharm_positions.shape}')
print(f'pharm direction (shape): {unconditional_samples[0].pharm_directions.shape}')


You can directly visualize samples with shepherd-score.

In [ ]:
draw_sample(unconditional_samples[0])

Convert molecules to rdkit molecules. By default, we relax samples with xTB and re-extract the interaction profile from there.

In [ ]:
mol_charges = mol_charges_from_samples(
    unconditional_samples,
    num_workers=10, # parallelize over 10 CPUs
    xtb_optimize=True, # extract the molecule after xTB relaxation
    xtb_timeout=60, # if the molecule cannot be optimized in 60 seconds, it will fail
)

mol_charges[0] # tuple of (rdkit.Chem.Mol, np.ndarray) where the np.ndarray is the partial charges

In [ ]:
mols = [mol for mol, _ in mol_charges if mol is not None]
Chem.Draw.MolsToGridImage(mols, molsPerRow=4)

In [ ]:
draw_mol(mols[0])